
## 심층 신경망(DNN) 깊게 이해하기: 구조 설계의 비밀

### 개요

Day 1에서는 PyTorch의 기본 요소를 배우고, 이를 바탕으로 첫 번째 신경망 모델을 성공적으로 구축했습니다. 

이제 우리는 한 걸음 더 나아가, 모델의 '성능'을 결정하는 핵심 요소인 `신경망의 구조(Architecture)` 에 대해 깊이 탐구해볼 시간입니다.

단순히 층을 쌓는 것을 넘어, "왜 더 깊게 쌓아야 하는가?", "각 층의 뉴런은 몇 개가 적절할까?", "가중치는 어떻게 시작해야 학습이 잘 될까?" 와 같은 근본적인 질문에 답을 찾아갑니다. 

또한, 모델의 안정적인 학습을 돕고 과적합을 방지하는 필수 테크닉인 `배치 정규화(Batch Normalization)` 와 `드롭아웃(Dropout)` 에 대해서도 배웁니다.

`이번 파트의 학습 목표:`

  * 모델의 깊이(은닉층)와 너비(뉴런 수)가 모델 성능에 미치는 영향을 이해하고, 이를 코드로 구현할 수 있습니다.
  
  * 가중치 초기화(Weight Initialization)의 중요성을 깨닫고, `Xavier`와 `He 초기화`를 모델에 적용할 수 있습니다.
  * \*\*배치 정규화(Batch Normalization)\*\*의 원리를 이해하고, 학습을 안정시키기 위해 모델에 적용할 수 있습니다.
  * 과적합을 방지하는 대표적인 규제 기법인 `드롭아웃(Dropout)` 을 모델에 적용할 수 있습니다.
  * 이 모든 기법을 종합하여 Day 1의 모델보다 훨씬 정교하고 강력한 분류 모델을 설계하고, 그 성능을 비교 분석할 수 있습니다.

이번 파트에서도 Day 1과 동일한 `위스콘신 유방암 데이터셋`을 사용하여, 우리가 배운 구조 설계 기법들이 실제로 모델 성능을 얼마나 향상시키는지 직접 확인해 보겠습니다.


### 1. 모델의 깊이와 너비: 은닉층(Hidden Layer)과 뉴런(Neuron)

모델의 성능은 그 구조에 크게 의존합니다. 구조를 결정하는 가장 기본적인 두 가지 요소는 '얼마나 깊게 쌓을 것인가'(깊이)와 '각 층을 얼마나 넓게 만들 것인가'(너비)입니다.

#### 1.1. 은닉층의 역할과 '깊이'

왜 우리는 여러 개의 은닉층을 쌓는 '깊은' 신경망을 만들까요?

단일 은닉층 모델이 이론적으로는 어떤 함수든 근사할 수 있다고 알려져 있지만, 이는 매우 '넓은'(뉴런 수가 아주 많은) 층을 필요로 합니다. 

반면, 층을 깊게 쌓으면 모델은 `계층적인 특징(Hierarchical Features)` 을 학습할 수 있게 됩니다.

예를 들어, 이미지 인식 모델의 초기 층은 간단한 엣지(edge)나 색상 패턴을 학습하고, 중간 층은 이들을 조합하여 눈, 코, 입과 같은 좀 더 복잡한 형태를 학습합니다. 

그리고 마지막 층은 이목구비의 조합을 통해 최종적으로 '얼굴'을 인식하게 됩니다. 

이처럼 깊은 구조는 데이터를 점진적으로 더 높은 수준의 추상적인 표현으로 변환해가며, 복잡한 문제를 훨씬 효율적으로 해결할 수 있도록 돕습니다.

  * `장점`: 더 적은 파라미터로 복잡한 함수를 표현할 수 있어 효율적입니다. 데이터의 계층적 구조를 학습할 수 있습니다.
  
  * `단점`: 너무 깊어지면 기울기 소실(Vanishing Gradient) 또는 폭주(Exploding Gradient) 문제가 발생하여 학습이 불안정해질 수 있습니다.

#### 1.2. 뉴런 수와 '너비'

층의 '너비', 즉 뉴런(또는 노드)의 수는 해당 층에서 얼마나 다양한 특징을 학습할지를 결정합니다. 뉴런 하나가 특정 패턴을 감지하는 역할을 한다고 생각할 수 있습니다.

  * `너비가 넓을수록(뉴런 수가 많을수록)`: 모델은 더 복잡하고 미세한 패턴을 학습할 수 있는 잠재력을 갖게 됩니다. 하지만, 이는 더 많은 계산량(파라미터 수 증가)을 요구하며, 데이터에 비해 모델이 너무 복잡해져 훈련 데이터에만 과도하게 최적화되는 `과적합(Overfitting)` 의 위험이 커집니다.
  
  * `너비가 좁을수록(뉴런 수가 적을수록)`: 모델이 단순해져 과적합의 위험은 줄어들지만, 데이터의 복잡한 패턴을 충분히 학습하지 못하는 `과소적합(Underfitting)` 이 발생할 수 있습니다.

따라서 깊이와 너비 사이의 적절한 균형을 찾는 것은 성공적인 모델 설계의 핵심 과제입니다.

#### 1.3. 코드 실습: 다양한 구조의 모델 비교하기

Day 1에서 만들었던 `SimpleClassifier`를 기반으로, 더 깊고 넓은 모델을 만들어보고 `torchsummary`를 통해 구조와 파라미터 수의 변화를 확인해 보겠습니다.

In [1]:
import torch
import torch.nn as nn
from torchsummary import summary
from sklearn.datasets import load_breast_cancer

# 데이터 로드 (입력 특성 개수 확인용)
X, y = load_breast_cancer(return_X_y=True)
input_features = X.shape[1]
output_classes = 2

# Day 1의 기본 모델
class SimpleClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super(SimpleClassifier, self).__init__()
        self.layer1 = nn.Linear(num_features, 16)
        self.layer2 = nn.Linear(16, 8)
        self.output_layer = nn.Linear(8, num_classes)
        self.relu = nn.ReLU()

    # sequnce 사용하면 불필요한 layer 과정 줄일 수 있음
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

In [2]:
# 더 깊은(Deep) 모델
class DeepClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super(DeepClassifier, self).__init__()
        self.layer1 = nn.Linear(num_features, 32)
        self.layer2 = nn.Linear(32, 16)
        self.layer3 = nn.Linear(16, 8) # 은닉층 추가
        self.output_layer = nn.Linear(8, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x)) # 추가된 층 통과
        x = self.output_layer(x)
        return x

In [3]:
# 더 넓은(Wide) 모델
class WideClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super(WideClassifier, self).__init__()
        # 각 층의 뉴런 수를 늘림
        self.layer1 = nn.Linear(num_features, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

In [5]:
#모델 인스턴스 생성
simple_model = SimpleClassifier(input_features, output_classes)
summary(simple_model, input_size=(input_features,))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 16]             496
              ReLU-2                   [-1, 16]               0
            Linear-3                    [-1, 8]             136
              ReLU-4                    [-1, 8]               0
            Linear-5                    [-1, 2]              18
Total params: 650
Trainable params: 650
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [6]:
deep_model = DeepClassifier(input_features, output_classes)
summary(deep_model, input_size=(input_features,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 32]             992
              ReLU-2                   [-1, 32]               0
            Linear-3                   [-1, 16]             528
              ReLU-4                   [-1, 16]               0
            Linear-5                    [-1, 8]             136
              ReLU-6                    [-1, 8]               0
            Linear-7                    [-1, 2]              18
Total params: 1,674
Trainable params: 1,674
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------


In [7]:
wide_model = WideClassifier(input_features, output_classes)
summary(wide_model, input_size=(input_features,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]           1,984
              ReLU-2                   [-1, 64]               0
            Linear-3                   [-1, 32]           2,080
              ReLU-4                   [-1, 32]               0
            Linear-5                    [-1, 2]              66
Total params: 4,130
Trainable params: 4,130
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.02
Estimated Total Size (MB): 0.02
----------------------------------------------------------------


### 2. 똑똑한 시작점 찾기: 가중치 초기화(Weight Initialization)

모델 학습은 '무작위'로 설정된 가중치에서 시작하여 점차 정답에 가까운 값으로 업데이트해나가는 과정입니다. 

이때, '무작위' 시작점을 얼마나 '똑똑하게' 정하느냐가 학습의 성패를 좌우할 수 있습니다. 

이것이 바로 `가중치 초기화`가 중요한 이유입니다.

잘못된 초기화는 학습 속도를 크게 저하시키거나, 심지어 학습이 전혀 진행되지 않는 문제를 야기합니다. (예: 기울기 소실/폭주)

<img src="https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FTzQod%2FbtsJ9upxYCd%2F743Oxp0zzA3DWZl1BWkVx1%2Fimg.png">


#### 2.1. Xavier(Glorot) 초기화

Xavier 초기화는 활성화 함수로 `Sigmoid`나 `tanh`를 사용할 때 주로 사용됩니다. 

이 방법의 핵심 아이디어는 `입력과 출력의 분산을 동일하게 유지`하는 것입니다. 

이를 통해 신호(활성값)가 여러 층을 통과하더라도 너무 커지거나 작아지는 것을 방지하여, 안정적인 역전파를 가능하게 합니다.

<img src="https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FbigENK%2FbtsJ9rs6cwE%2FkApxcczmRKQ2AOmu5q2Kc1%2Fimg.png">


#### 2.2. He 초기화

`ReLU` 활성화 함수는 양수 값을 그대로 통과시키고 음수 값은 0으로 만듭니다. 

이 특성 때문에 Xavier 초기화를 사용하면 층을 거칠수록 활성값의 분산이 절반으로 줄어드는 경향이 있습니다.

`He 초기화`는 이러한 ReLU의 특성을 고려하여, 출력 분산이 입력 분산의 절반이 되도록 가중치를 초기화합니다. 

<img src="https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FbiwrzN%2FbtsKaLw6nAP%2FYEgJ5m4AOuGwFBIDs2lYv1%2Fimg.png">

결과적으로 여러 ReLU 층을 통과하더라도 신호가 죽지 않고 잘 전달되어, 깊은 네트워크의 학습을 효과적으로 돕습니다. 

현대의 `ReLU` 기반 딥러닝 모델에서는 He 초기화가 표준처럼 사용됩니다.

#### 2.3. 코드 실습: 모델에 가중치 초기화 적용하기

PyTorch에서는 `torch.nn.init` 모듈을 통해 다양한 초기화 기법을 손쉽게 적용할 수 있습니다. 

`model.apply()` 함수를 사용하면 모델의 모든 모듈(계층)을 순회하며 특정 함수를 적용할 수 있습니다.

In [12]:
import torch
import torch.nn as nn
import plotly.express as px

# He 초기화를 적용할 모델 정의
class ModelWithHeInit(nn.Module):
    def __init__(self, num_features, num_classes):
        super(ModelWithHeInit, self).__init__()
        self.layer1 = nn.Linear(num_features, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output_layer = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        # 모델 생성 시 가중치 초기화 함수 적용
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # Linear 계층인지 확인
        if isinstance(module, nn.Linear):
            # He 초기화 (kaiming_normal_) 적용ㅁ
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            # 편향(bias)은 0으로 초기화
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

In [13]:
# 기본 초기화 모델과 He 초기화 모델 생성
default_model = WideClassifier(input_features, output_classes)
he_init_model = ModelWithHeInit(input_features, output_classes)

In [14]:
# 각 모델의 첫 번째 레이어 가중치 분포 시각화
default_weights = default_model.layer1.weight.detach().numpy().flatten()
he_init_weights = he_init_model.layer1.weight.detach().numpy().flatten()

fig = px.histogram(x=default_weights, nbins=100, title='Default Initialization Weight Distribution')
fig.show()

fig = px.histogram(x=he_init_weights, nbins=100, title='He Initialization Weight Distribution')
fig.show()

### 3. 학습 안정화 (1): 배치 정규화(Batch Normalization)

깊은 모델을 학습시킬 때 마주하는 또 다른 난관은 `내부 공변량 변화(Internal Covariate Shift)` 입니다. 

이는 훈련 과정에서 이전 층의 파라미터가 변하면서, 현재 층에 들어오는 입력 데이터의 분포가 계속 바뀌는 현상을 말합니다. 

롤러코스터처럼 시시각각 변하는 땅 위에서 목표물을 맞추려는 것과 같습니다.

`배치 정규화(Batch Normalization)`는 이 문제를 해결하기 위해 제안되었습니다. 

각 미니배치(mini-batch) 데이터에 대해, 계층의 활성화 함수를 통과하기 전 값들을 `평균 0, 분산 1인 정규분포`로 만듭니다. 

그리고 나서, 모델이 표현력을 잃지 않도록 새로운 파라미터($\gamma$: 크기 조절, $\beta$: 이동)를 학습하여 정규화된 데이터의 스케일과 시프트를 조정합니다.

`배치 정규화의 효과:`

  * `학습 속도 향상`: 내부 공변량 변화를 줄여 학습을 안정화시키고, 더 높은 학습률(learning rate)을 사용할 수 있게 해줍니다.
  
  * `규제(Regularization) 효과`: 각 미니배치의 평균과 분산을 사용하므로 약간의 노이즈가 추가되는 효과가 있어, 과적합을 억제하는 데 도움이 됩니다.
  * `기울기 소실 문제 완화`: 활성값이 특정 범위에 치우치는 것을 막아 기울기 소실 문제를 완화합니다.

#### 3.4. 코드 실습: 모델에 배치 정규화 적용하기

PyTorch에서는 `nn.BatchNorm1d` (1차원 데이터, 즉 일반적인 DNN용) 또는 `nn.BatchNorm2d`(이미지용) 레이어를 추가하여 배치 정규화를 적용합니다. 

보통 `선형 계층(Linear Layer)과 활성화 함수(Activation Function) 사이`에 위치시킵니다.

In [15]:
import torch.nn as nn

class ModelWithBN(nn.Module):
    def __init__(self, num_features, num_classes):
        super(ModelWithBN, self).__init__()
        self.layer1 = nn.Linear(num_features, 64)
        self.bn1 = nn.BatchNorm1d(64) # Linear layer의 출력 뉴런 수와 동일하게 설정
        self.layer2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.output_layer = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Linear -> BatchNorm -> Activation 순서
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu(x)

        x = self.output_layer(x)
        return x

bn_model = ModelWithBN(input_features, output_classes)
print("===== Model with Batch Normalization =====")
summary(bn_model, input_size=(input_features,))

===== Model with Batch Normalization =====
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]           1,984
       BatchNorm1d-2                   [-1, 64]             128
              ReLU-3                   [-1, 64]               0
            Linear-4                   [-1, 32]           2,080
       BatchNorm1d-5                   [-1, 32]              64
              ReLU-6                   [-1, 32]               0
            Linear-7                    [-1, 2]              66
Total params: 4,322
Trainable params: 4,322
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.02
Estimated Total Size (MB): 0.02
----------------------------------------------------------------


`중요`: 배치 정규화는 훈련 시와 평가 시에 다르게 동작합니다. 

훈련 시에는 현재 미니배치의 평균/분산을 사용하지만, 평가 시에는 훈련 과정에서 이동 평균(moving average)으로 계산해 둔 전체 데이터의 평균/분산을 사용합니다. 

따라서 모델을 평가할 때는 반드시 `model.eval()` 모드로 전환해야 합니다.

### 4. 과적합 방지: 드롭아웃(Dropout)

`과적합(Overfitting)`은 모델이 훈련 데이터는 거의 완벽하게 예측하지만, 본 적 없는 새로운 데이터(테스트 데이터)에 대해서는 성능이 떨어지는 현상입니다. 

모델이 데이터의 실제 패턴이 아닌, 훈련 데이터에만 존재하는 노이즈까지 암기해버리는 것이 원인입니다.

`드롭아웃(Dropout)`은 이를 해결하기 위한 가장 간단하고 강력한 규제 기법 중 하나입니다. 

훈련 과정에서 각 뉴런을 `지정된 확률(p)로 무작위하게 비활성화(출력을 0으로 만듦)`시키는 방식입니다.

이는 마치 매번 다른 구성의 '더 작은' 신경망 여러 개를 동시에 학습시키는 것과 같은 효과를 냅니다. 

특정 뉴런에 과도하게 의존할 수 없게 되므로, 모델은 더 강건하고 일반화된 특징을 학습하게 됩니다. 이를 `앙상블(Ensemble) 효과`라고 합니다.

#### 4.1. 코드 실습: 모델에 드롭아웃 적용하기

PyTorch에서는 `nn.Dropout(p)` 레이어를 추가하여 드롭아웃을 적용합니다. 보통 `활성화 함수(Activation Function) 뒤`에 위치시킵니다.

In [16]:
import torch.nn as nn

class ModelWithDropout(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.5):
        super(ModelWithDropout, self).__init__()
        self.layer1 = nn.Linear(num_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.layer2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.output_layer = nn.Linear(32, num_classes)

        self.relu = nn.ReLU()
        # 드롭아웃 레이어 정의 (p=0.5는 50%의 뉴런을 비활성화)
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        # Linear -> BatchNorm -> Activation -> Dropout 순서
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.output_layer(x)
        return x

dropout_model = ModelWithDropout(input_features, output_classes)
print("===== Model with Dropout =====")
summary(dropout_model, input_size=(input_features,))

===== Model with Dropout =====
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]           1,984
       BatchNorm1d-2                   [-1, 64]             128
              ReLU-3                   [-1, 64]               0
           Dropout-4                   [-1, 64]               0
            Linear-5                   [-1, 32]           2,080
       BatchNorm1d-6                   [-1, 32]              64
              ReLU-7                   [-1, 32]               0
           Dropout-8                   [-1, 32]               0
            Linear-9                    [-1, 2]              66
Total params: 4,322
Trainable params: 4,322
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.02
Estimated Total Size (MB): 0.02
----------------

`중요`: 드롭아웃 역시 배치 정규화처럼 훈련 시에만 적용되어야 합니다. 

`model.eval()` 모드로 전환하면 드롭아웃은 자동으로 비활성화됩니다.

따라서 `model.train()`과 `model.eval()`의 올바른 사용은 매우 중요합니다.


### 5. 종합 실습: 더 강력한 분류 모델 만들기

이제 오늘 배운 모든 기법(깊은 구조, He 초기화, 배치 정규화, 드롭아웃)을 총동원하여 '고급 분류 모델'을 만들고, Day 1의 기본 모델과 성능을 비교해보겠습니다.

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import plotly.graph_objects as go

In [17]:
# 0. 데이터 준비
X, y = load_breast_cancer(return_X_y=True)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 데이터 스케일링
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# Dataset 및 DataLoader
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features) # 실수형으로 정리
        self.labels = torch.LongTensor(labels) # 정수형으로 정리
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [19]:
# 1. 고급 분류 모델 (Advanced Classifier) 정의
class AdvancedClassifier(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.4):
        super(AdvancedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.net(x)

In [20]:
# 2. 모델 학습 및 평가 함수
def train_and_evaluate(model, train_loader, test_loader, num_epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001) # 가중치 없데이트 (최근에 Adam 많이 사용)

    history = {'train_loss': [], 'test_loss': [], 'test_accuracy': []}

    for epoch in range(num_epochs):
        # --- 훈련 ---
        model.train() # 모델을 훈련 모드로 설정
        epoch_train_loss = 0.0 # 훈련 손실 초기화
        for features, labels in train_loader: # 32개 레코드에 대해 x, y 값을 내보내줌
            features, labels = features.to(device), labels.to(device) # torch에서는 모델을 학습할 때 데이터를 GPU로 보내야 함 (CPU에서 학습하면 속도가 느림)
            optimizer.zero_grad() # 기울기 초기화
            outputs = model(features) # 모델 출력 계산
            loss = criterion(outputs, labels) # 손실 계산
            loss.backward() # 역전파
            optimizer.step() # 가중치 업데이트
            epoch_train_loss += loss.item() # 훈련 손실 누적

            # 위 과정이 딥러닝 - 이게 각 배치마다 일어남
            # 평가는 1 epoch마다 하게 됨

        # --- 평가 ---
        model.eval() # 모델을 평가 모드로 설정
        epoch_test_loss = 0.0 # 평가 손실 초기화
        correct = 0 # 예측 정확도
        total = 0 # 전체 데이터 수
        with torch.no_grad(): # 평가 모드에서는 기울기 계산 필요 없음
            for features, labels in test_loader:
                features, labels = features.to(device), labels.to(device) # 데이터를 GPU로 보냄
                outputs = model(features) # 모델 출력 계산
                loss = criterion(outputs, labels) # 손실 계산
                epoch_test_loss += loss.item() # 평가 손실 누적

                _, predicted = torch.max(outputs.data, 1) # 예측 클래스 계산 ( 1: 행 방향으로 최대값 찾기, 0: 열 방향으로 최대값 찾기)
                total += labels.size(0) # 전체 데이터 수 누적
                correct += (predicted == labels).sum().item() # 예측 정확도 누적

        train_loss = epoch_train_loss / len(train_loader) # 훈련 손실 평균
        test_loss = epoch_test_loss / len(test_loader) # 평가 손실 평균
        test_accuracy = 100 * correct / total # 평가 정확도

        history['train_loss'].append(train_loss) # 훈련 손실 기록
        history['test_loss'].append(test_loss) # 평가 손실 기록
        history['test_accuracy'].append(test_accuracy) # 평가 정확도 기록

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%')

    return history

In [21]:
# 3. 기본 모델과 고급 모델 학습 및 비교
print("--- Training Simple Model ---")
simple_model = SimpleClassifier(input_features, output_classes)
simple_history = train_and_evaluate(simple_model, train_loader, test_loader)

print("--- Training Advanced Model ---")
advanced_model = AdvancedClassifier(input_features, output_classes)
advanced_history = train_and_evaluate(advanced_model, train_loader, test_loader)

--- Training Simple Model ---
Epoch [10/50], Train Loss: 0.1298, Test Loss: 0.1517, Test Accuracy: 95.61%
Epoch [20/50], Train Loss: 0.0595, Test Loss: 0.0939, Test Accuracy: 96.49%
Epoch [30/50], Train Loss: 0.0458, Test Loss: 0.0811, Test Accuracy: 96.49%
Epoch [40/50], Train Loss: 0.0388, Test Loss: 0.0798, Test Accuracy: 95.61%
Epoch [50/50], Train Loss: 0.0318, Test Loss: 0.0848, Test Accuracy: 95.61%
--- Training Advanced Model ---
Epoch [10/50], Train Loss: 0.2090, Test Loss: 0.1262, Test Accuracy: 97.37%
Epoch [20/50], Train Loss: 0.1706, Test Loss: 0.0956, Test Accuracy: 95.61%
Epoch [30/50], Train Loss: 0.1188, Test Loss: 0.0752, Test Accuracy: 97.37%
Epoch [40/50], Train Loss: 0.1100, Test Loss: 0.0728, Test Accuracy: 96.49%
Epoch [50/50], Train Loss: 0.0814, Test Loss: 0.0684, Test Accuracy: 97.37%


In [22]:
# 4. 결과 시각화
fig = go.Figure()
epochs = list(range(1, 51))
# Simple Model
fig.add_trace(go.Scatter(x=epochs, y=simple_history['train_loss'], name='Simple - Train Loss', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=epochs, y=simple_history['test_loss'], name='Simple - Test Loss'))
# Advanced Model
fig.add_trace(go.Scatter(x=epochs, y=advanced_history['train_loss'], name='Advanced - Train Loss', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=epochs, y=advanced_history['test_loss'], name='Advanced - Test Loss'))

fig.update_layout(title='Simple vs Advanced Model Loss', xaxis_title='Epochs', yaxis_title='Loss')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=simple_history['test_accuracy'], name='Simple Model Accuracy'))
fig.add_trace(go.Scatter(x=epochs, y=advanced_history['test_accuracy'], name='Advanced Model Accuracy'))
fig.update_layout(title='Simple vs Advanced Model Accuracy', xaxis_title='Epochs', yaxis_title='Accuracy (%)')
fig.show()

두 모델 모두 학습이 진행될수록 손실이 감소하는 경향을 보이며, Advanced 모델은 더 복잡한 구조로 인해 훈련 손실이 더 낮게 나타나지만, 

평가 손실에서는 오히려 변동이 크거나 단순 모델과 큰 차이가 없을 수 있습니다.  

이러한 결과는 모델의 복잡도가 높아질수록 훈련 데이터에 더 잘 맞추지만, 과적합(overfitting) 가능성도 함께 증가할 수 있음을 시사합니다.